# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to use the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library to explore the FAIR^2 dataset, as defined by its Croissant schema. We will:
- Load metadata and data records
- Explore record sets, fields, and values using their `@id`
- Extract and analyze data
- Visualize and summarize findings

### Dataset Source
- Schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)
- FAIR^2 identifier: `10.71728/senscience.vq4a-28xa`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load dataset metadata using `mlcroissant`. The schema describes dataset structure, record sets, and fields using unique `@id` references.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata (this fetches and parses the Croissant schema)
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Name: {getattr(metadata, 'name', '')}\nDescription: {getattr(metadata, 'description', '')}")

## 2. Data Overview
List record sets and their constituent fields using `@id`. This ensures that all further references use the unique identifiers specified in the schema.

In [ ]:
# List all record sets by their @id
print('Record Sets:')
record_sets = list(metadata.record_sets)
record_set_infos = []

for rs in record_sets:
    print(f"  - @id: {getattr(rs, '@id', rs)}  | name: {getattr(rs, 'name', '')}")
    fields = list(getattr(rs, 'fields', []))
    print("    Fields:")
    for f in fields:
        print(f"      • @id: {getattr(f, '@id', f)} | name: {getattr(f, 'name', '')}")
    record_set_infos.append({
        'id': getattr(rs, '@id', rs),
        'fields': [getattr(f, '@id', f) for f in fields]
    })

if not record_sets:
    print("  (No explicit record sets listed in dataset metadata. Attempting to enumerate tabular files as record sets.)")

# If there are no record_sets in metadata, try to enumerate the record_set from the dataset distribution
if len(record_sets) == 0:
    # dataset.metadata.distribution should contain DataFileObject(s) with @id
    print("Distributions:")
    for dist in getattr(metadata, 'distribution', []):
        print(f"  - @id: {getattr(dist, '@id', dist)}  | name: {getattr(dist, 'name', '')}  | encodingFormat: {getattr(dist, 'encoding_format', getattr(dist, 'encodingFormat', ''))}")

## 3. Data Extraction
Load data from a selected record set using its `@id` into a Pandas DataFrame for further analysis. All fields (columns) are referenced by their `@id` per the Croissant standard.

If the previous step shows no explicit record sets, use the available data file (distribution) as the record set.

In [ ]:
# Identify the appropriate record set to use.
if record_sets:
    # Use the first declared record set
    selected_record_set_id = getattr(record_sets[0], '@id', record_sets[0])
    print(f'Selected record set @id: {selected_record_set_id}')
else:
    # Fall back to the first distribution as a record set
    first_dist = getattr(metadata, 'distribution', [])[0]
    selected_record_set_id = getattr(first_dist, '@id', first_dist)
    print(f'Selected fallback record set (from distribution) @id: {selected_record_set_id}')

# Load the records for the selected record set
try:
    records = list(dataset.records(record_set=selected_record_set_id))
    df = pd.DataFrame(records)
    print(f"Loaded {df.shape[0]} records with columns:")
    print(df.columns.tolist())
    display(df.head())
except Exception as e:
    print(f"Error loading records for record set {selected_record_set_id}: {e}")
    df = None

## 4. Exploratory Data Analysis (EDA)
Apply standard data processing steps, referencing columns by their `@id` (as used in the DataFrame above).

- Select a numeric field using its `@id`
- Filter out records below a threshold
- Normalize the data (z-score)
- Optionally, group by a categorical attribute

In [ ]:
import numpy as np
if df is not None:
    # Try to infer a numeric field @id, else select the first appropriate column
    numeric_candidate_ids = [col for col in df.columns if np.issubdtype(df[col].dropna().dtype, np.number)]
    if numeric_candidate_ids:
        numeric_field_id = numeric_candidate_ids[0]
    else:
        # Try to convert common numeric columns to numeric
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
                if np.issubdtype(df[col].dtype, np.number):
                    numeric_field_id = col
                    break
            except:
                continue
        else:
            numeric_field_id = df.columns[0]
    print(f"Using numeric field @id for EDA: {numeric_field_id}")

    # Filter
    threshold = df[numeric_field_id].mean() if np.issubdtype(df[numeric_field_id].dtype, np.number) else 0
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Select a grouping field if available (prefer a non-numeric)
    group_field_id = None
    non_numeric_fields = [col for col in df.columns if not np.issubdtype(df[col].dropna().dtype, np.number)]
    if non_numeric_fields:
        group_field_id = non_numeric_fields[0]

    if group_field_id is not None:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
    else:
        print("No suitable grouping field detected.")

## 5. Visualization
Let's plot the distribution of the selected numeric field and, if groupings are available, visualize them as well.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id], bins=30, kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()

    if group_field_id is not None and group_field_id in df.columns:
        plt.figure(figsize=(8, 5))
        order = df[group_field_id].value_counts().index
        sns.boxplot(y=group_field_id, x=numeric_field_id, data=df, order=order)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()
else:
    print("No data loaded for visualization.")

## 6. Conclusion

- We demonstrated how to load and explore dataset metadata and records using [`mlcroissant`](https://pypi.org/project/mlcroissant/) and referenced all data by their Croissant `@id`.
- The workflow included listing available record sets/fields, loading records, basic EDA (filtering, normalization, grouping), and visualization.
- For production analysis, further domain-specific filtering and annotation can be applied based on the field `@id` documented in the dataset's Croissant schema.